 # AAL Sales Analysis - Q4 2020
This notebook analyzes the Q4 2020 sales data for AAL (Australian Apparel Limited) across different states, demographic groups, and time periods.

In [17]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

Step1: Data Loading

In [18]:
df = pd.read_csv('AusApparalSales4thQrt2020.csv')
df.head()

,Date,Time,State,Group,Unit,Sales
0,1-Oct-2020,Morning,WA,Kids,8,20000
1,1-Oct-2020,Morning,WA,Men,8,20000
2,1-Oct-2020,Morning,WA,Women,4,10000
3,1-Oct-2020,Morning,WA,Seniors,15,37500
4,1-Oct-2020,Afternoon,WA,Kids,3,7500


Step 2: Data Inspection

In [19]:
print(f"DataFrame shape: {df.shape}")
print("\nColumn types:")
print(df.dtypes)
print("\nInfo:")
print(df.info())


DataFrame shape: (7560, 6)

Column types:
Date       str
Time       str
State      str
Group      str
Unit     int64
Sales    int64
dtype: object

Info:
<class 'pandas.DataFrame'>
RangeIndex: 7560 entries, 0 to 7559
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    7560 non-null   str  
 1   Time    7560 non-null   str  
 2   State   7560 non-null   str  
 3   Group   7560 non-null   str  
 4   Unit    7560 non-null   int64
 5   Sales   7560 non-null   int64
dtypes: int64(2), str(4)
memory usage: 354.5 KB
None


Step 3: Data Cleaning

In [20]:
# Remove whitespaces from column names
df.columns = df.columns.str.strip()

# Remove whitespaces from string columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

print("Columns:", df.columns.tolist())
print("\nSample data:")
print(df.head())

Columns: ['Date', 'Time', 'State', 'Group', 'Unit', 'Sales']

Sample data:
         Date       Time State    Group  Unit  Sales
0  1-Oct-2020    Morning    WA     Kids     8  20000
1  1-Oct-2020    Morning    WA      Men     8  20000
2  1-Oct-2020    Morning    WA    Women     4  10000
3  1-Oct-2020    Morning    WA  Seniors    15  37500
4  1-Oct-2020  Afternoon    WA     Kids     3   7500


C:\Users\kpilgulwar\AppData\Local\Temp\ipykernel_10020\2125280494.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [21]:
# Check for missing values
print("\nMissing values (isna):")
print(df.isna().sum())
print("\nMissing values (notna):")
print(df.notna().sum())


Missing values (isna):
Date     0
Time     0
State    0
Group    0
Unit     0
Sales    0
dtype: int64

Missing values (notna):
Date     7560
Time     7560
State    7560
Group    7560
Unit     7560
Sales    7560
dtype: int64


### Recommendation for Handling Missing Data

 **Observation:** The dataset has **no missing values** in any column — all 7560 records are complete

 **Recommendation:**
 For this dataset, since there are no missing values, no treatment is needed. However, `dropna()` is still included in the code as a safety step in case the data changes in the future.

In [22]:
# drop rows with missing values
df = df.dropna()

In [12]:
# Dealing with duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

# Data Cleaning
df['Date'] = pd.to_datetime(df['Date'])



Duplicate rows: 0


###  Data Normalization

  **Why Normalization over Standardization?**
  - **Normalization** (Min-Max scaling) scales data to a 0–1 range. Best when data does not follow a normal distribution.
  - **Standardization** (Z-score) centers data around mean=0, std=1. Best for normally distributed data.

Since sales data is often skewed (a few states sell much more), **normalization** is preferred here — it preserves the relative differences and is easier to interpret (0 = lowest, 1 = highest).

In [23]:
# min max normalization on 'Sales' column
df['Sales_normalized'] = (df['Sales'] - df['Sales'].min()) / (df['Sales'].max() - df['Sales'].min())

# print sales and normalized sales
print("\nSales and Normalized Sales:")
print(df[['Sales', 'Sales_normalized']].head())


Sales and Normalized Sales:
   Sales  Sales_normalized
0  20000          0.095238
1  20000          0.095238
2  10000          0.031746
3  37500          0.206349
4   7500          0.015873


## __10. Aggregating Data__
GroupBy Insights

  **GroupBy** splits data into groups based on a column, applies a function (sum, mean, etc.), and combines the results. This is
  essential for:
  - **Data chunking:** Breaking sales data by State or Group to analyze each segment independently.
  - **Data merging/aggregation:** Summarizing total sales per state, average units per group, etc.

  **Recommendation:** Use GroupBy for **aggregation** in this analysis — it lets us compare total and average sales across states
  and demographic groups, which directly answers the CEO's questions about revenue by state.

In [24]:
print("State wise total sales:")
state_sales = df.groupby('State')[['Sales', 'Unit']].sum().sort_values(by='Sales', ascending=False)
print(state_sales)

State wise total sales:
           Sales   Unit
State                  
VIC    105565000  42226
NSW     74970000  29988
SA      58857500  23543
QLD     33417500  13367
TAS     22760000   9104
NT      22580000   9032
WA      22152500   8861


In [25]:
# Group wise total sales
print("\nGroup wise total sales:")
group_sales = df.groupby('Group')[['Sales', 'Unit']].sum().sort_values(by='Sales', ascending=False)
print(group_sales)


Group wise total sales:
            Sales   Unit
Group                   
Men      85750000  34300
Women    85442500  34177
Kids     85072500  34029
Seniors  84037500  33615


In [27]:
print("=== State-wise Total Sales ===")
state_sales = df.groupby('State')[['Unit', 'Sales', 'Sales_normalized']].sum().sort_values('Sales', ascending=False)
print(state_sales)

print("\n=== Group-wise Total Sales ===")
group_sales = df.groupby('Group')[['Unit', 'Sales', 'Sales_normalized']].sum().sort_values('Sales', ascending=False)
print(group_sales)

print("\n=== State & Group wise Average Sales ===")
state_group_sales = df.groupby(['State', 'Group'])[['Unit', 'Sales', 'Sales_normalized']].mean()
print(state_group_sales)

=== State-wise Total Sales ===
        Unit      Sales  Sales_normalized
State                                    
VIC    42226  105565000        635.968254
NSW    29988   74970000        441.714286
SA     23543   58857500        339.412698
QLD    13367   33417500        177.888889
TAS     9104   22760000        110.222222
NT      9032   22580000        109.079365
WA      8861   22152500        106.365079

=== Group-wise Total Sales ===
          Unit     Sales  Sales_normalized
Group                                     
Men      34300  85750000        484.444444
Women    34177  85442500        482.492063
Kids     34029  85072500        480.142857
Seniors  33615  84037500        473.571429

=== State & Group wise Average Sales ===
                    Unit         Sales  Sales_normalized
State Group                                             
NSW   Kids     27.537037  68842.592593          0.405350
      Men      28.181481  70453.703704          0.415579
      Seniors  26.944444  67361